In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

import pandas as pd

factor = 1
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

# load just the datasets q01 needs:
def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

# Define the cutoff date as a pandas Timestamp (will be handled on GPU)
date = pd.Timestamp("1995-03-04")
fcustomer = customer.loc[
    customer.C_MKTSEGMENT == "HOUSEHOLD",
    ["C_CUSTKEY"]
]

forders = orders.loc[
    orders.O_ORDERDATE < date,
    ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_SHIPPRIORITY"]
]

flineitem = lineitem.loc[
    lineitem.L_SHIPDATE > date,
    ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
]

# 2) Perform the two joins entirely on GPU
jn2 = (
    fcustomer
      .merge(forders,    left_on="C_CUSTKEY", right_on="O_CUSTKEY")
      .merge(flineitem,  left_on="O_ORDERKEY", right_on="L_ORDERKEY")
)

# 3) Compute the TMP column, group, sum and sort—all GPU operations
jn2["TMP"] = jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT)

res = (
    jn2
      .groupby(
          ["L_ORDERKEY", "O_ORDERDATE", "O_SHIPPRIORITY"],
          as_index=False,
          sort=False
      )["TMP"]
      .sum()
      .sort_values("TMP", ascending=False)
)